In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 904.3/904.3 kB 39.6 MB/s eta 0:00:00


In [13]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("/content/highway_mini.mp4")
assert cap.isOpened(), "Error reading video file"
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

# Video writer
video_writer = cv2.VideoWriter("highway3333.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Define speed region points
speed_region = [(20, 400), (1800, 400), (1800, 360), (20, 360)]

speed = solutions.SpeedEstimator(
    show=True,  # Display the output
    model="yolo11n.pt",  # Path to the YOLO11 model file.
    region=speed_region,  # Pass region points
    #classes=[0, 2],  # If you want to estimate speed of specific classes.
    # line_width=2,  # Adjust the line width for bounding boxes and text display
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break
    out = speed.estimate_speed(im0)
    video_writer.write(im0)

cap.release()
video_writer.release()
cv2.destroyAllWindows()

Ultralytics Solutions: ✅ {'region': [(20, 400), (2000, 400), (2000, 360), (20, 360)], 'show_in': True, 'show_out': True, 'colormap': None, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'json_file': None, 'records': 5, 'show': True, 'model': 'yolo11n.pt', 'classes': [0, 2]}
WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()


0: 384x640 10 cars, 240.5ms
Speed: 4.0ms preprocess, 240.5ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 221.3ms
Speed: 7.7ms preprocess, 221.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 197.9ms
Speed: 4.2ms preprocess, 197.9ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 159.0ms
Speed: 3.9ms preprocess, 159.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 117.2ms
Speed: 3.8ms preprocess, 117.2ms inference, 1.5ms postprocess per image at sha

modify code from the above but for calculation of speed

In [14]:
import cv2
from ultralytics import solutions
import numpy as np

# Open video capture
cap = cv2.VideoCapture("/content/highway_mini.mp4")
assert cap.isOpened(), "Error reading video file"
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

# Resize factor (optional, adjust based on your needs)
resize_factor = 0.75  # Resize frames for improved accuracy at a cost of speed

# Video writer setup
video_writer = cv2.VideoWriter("highway444444.avi", cv2.VideoWriter_fourcc(*"XVID"), fps, (int(w * resize_factor), int(h * resize_factor)))

# Define speed region points
speed_region = [(20, 400), (1800, 400), (1800, 360), (20, 360)]

# Initialize SpeedEstimator
speed = solutions.SpeedEstimator(
    show=True,  # Display the output
    model="yolo11n.pt",  # Path to the YOLO11 model file.
    region=speed_region,  # Define region of interest for speed estimation
    classes=[2],  # Specify class(es) to estimate speed (e.g., vehicles)
)

# Function for smoothing speed estimates (e.g., moving average)
def smooth_speed(speed_estimates, window_size=5):
    if len(speed_estimates) < window_size:
        return np.mean(speed_estimates)  # Return average of available estimates
    return np.mean(speed_estimates[-window_size:])  # Apply moving average on last `window_size` estimates

# Function to convert pixel distance to real-world distance (assuming you know the real size of an object)
def convert_to_real_world_distance(pixel_distance, pixels_per_meter):
    return pixel_distance / pixels_per_meter

# List to store speed estimates for smoothing
speed_estimates = []
frame_count = 0  # Counter for frame processing
prev_positions = {}  # Track previous positions of objects
while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    # Resize frame for faster processing
    im0_resized = cv2.resize(im0, (int(w * resize_factor), int(h * resize_factor)))

    # Estimate speed
    out = speed.estimate_speed(im0_resized)

    # Extract object position and velocity info
    if "objects" in out:  # Assuming that 'objects' contains detection results
        for obj in out["objects"]:
            object_id = obj["id"]
            x1, y1, x2, y2 = obj["bbox"]  # Bounding box coordinates

            # Assuming you know real-world scale (meters per pixel)
            real_car_length = 4  # meters
            pixels_per_meter = real_car_length / (x2 - x1)

            # Calculate pixel distance traveled between frames (considering previous position)
            if object_id in prev_positions:
                prev_x, prev_y = prev_positions[object_id]
                pixel_distance = np.sqrt((x2 - prev_x)**2 + (y2 - prev_y)**2)
                real_distance = convert_to_real_world_distance(pixel_distance, pixels_per_meter)

                # Calculate time interval between frames (seconds)
                time_interval = 1 / fps  # FPS is the frame rate of the video
                speed = real_distance / time_interval  # Speed in meters per second

                # Append current speed for smoothing
                speed_estimates.append(speed)

            # Save current position for next frame
            prev_positions[object_id] = (x2, y2)

    # Apply smoothing to the speed estimates
    smoothed_speed = smooth_speed(speed_estimates)

    # Display smoothed speed on frame
    cv2.putText(im0_resized, f"Speed: {smoothed_speed:.2f} m/s", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Write processed frame to output video
    video_writer.write(im0_resized)

    frame_count += 1

# Release resources
cap.release()
video_writer.release()
cv2.destroyAllWindows()


Ultralytics Solutions: ✅ {'region': [(20, 400), (1800, 400), (1800, 360), (20, 360)], 'show_in': True, 'show_out': True, 'colormap': None, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'json_file': None, 'records': 5, 'show': True, 'model': 'yolo11n.pt', 'classes': [2]}
WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()


0: 384x640 9 cars, 125.7ms
Speed: 2.9ms preprocess, 125.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)



/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


0: 384x640 9 cars, 184.2ms
Speed: 4.8ms preprocess, 184.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 208.2ms
Speed: 4.5ms preprocess, 208.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 173.3ms
Speed: 4.3ms preprocess, 173.3ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 172.9ms
Speed: 4.2ms preprocess, 172.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 170.1ms
Speed: 4.3ms preprocess, 170.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 179.4ms
Speed: 4.0ms preprocess, 179.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 176.9ms
Speed: 4.1ms preprocess, 176.9ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 186.3ms
Speed: 4.3ms preprocess, 186.3ms inference, 1.7ms postprocess per image at shape (1, 

Modify code from the above and add speed in 100+

In [16]:
import cv2
from ultralytics import solutions
import numpy as np

# Open video capture
cap = cv2.VideoCapture("/content/highway_mini.mp4")
assert cap.isOpened(), "Error reading video file"
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

# Resize factor (optional, adjust based on your needs)
resize_factor = 0.75  # Resize frames for improved accuracy at a cost of speed

# Video writer setup
video_writer = cv2.VideoWriter("highway333300.avi", cv2.VideoWriter_fourcc(*"XVID"), fps, (int(w * resize_factor), int(h * resize_factor)))

# Define speed region points
speed_region = [(20, 400), (1800, 400), (1800, 360), (20, 360)]

# Initialize SpeedEstimator
speed = solutions.SpeedEstimator(
    show=True,  # Display the output
    model="yolo11n.pt",  # Path to the YOLO11 model file.
    region=speed_region,  # Define region of interest for speed estimation
    classes=[2],  # Specify class(es) to estimate speed (e.g., vehicles)
)

# Function for smoothing speed estimates (e.g., moving average)
def smooth_speed(speed_estimates, window_size=5):
    if len(speed_estimates) < window_size:
        return np.mean(speed_estimates)  # Return average of available estimates
    return np.mean(speed_estimates[-window_size:])  # Apply moving average on last `window_size` estimates

# Function to convert pixel distance to real-world distance (assuming you know the real size of an object)
def convert_to_real_world_distance(pixel_distance, pixels_per_meter):
    return pixel_distance / pixels_per_meter

# List to store speed estimates for smoothing
speed_estimates = []
frame_count = 0  # Counter for frame processing
prev_positions = {}  # Track previous positions of objects

# Define real-world scaling (adjust this according to your setup)
real_car_length = 4  # meters (example: the length of a car)
car_pixel_length = 100  # pixels (adjust this by measuring the car's length in pixels)
pixels_per_meter = car_pixel_length / real_car_length  # Scaling factor

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    # Resize frame for faster processing
    im0_resized = cv2.resize(im0, (int(w * resize_factor), int(h * resize_factor)))

    # Estimate speed
    out = speed.estimate_speed(im0_resized)

    # Extract object position and velocity info
    if "objects" in out:  # Assuming that 'objects' contains detection results
        for obj in out["objects"]:
            object_id = obj["id"]
            x1, y1, x2, y2 = obj["bbox"]  # Bounding box coordinates

            # Calculate pixel distance traveled between frames (considering previous position)
            if object_id in prev_positions:
                prev_x, prev_y = prev_positions[object_id]
                pixel_distance = np.sqrt((x2 - prev_x)**2 + (y2 - prev_y)**2)
                real_distance = convert_to_real_world_distance(pixel_distance, pixels_per_meter)

                # Calculate time interval between frames (seconds)
                time_interval = 1 / fps  # FPS is the frame rate of the video
                speed = real_distance / time_interval  # Speed in meters per second

                # Convert speed from m/s to km/h
                speed_kmh = speed * 3.6

                # Append current speed for smoothing
                speed_estimates.append(speed_kmh)

            # Save current position for next frame
            prev_positions[object_id] = (x2, y2)

    # Apply smoothing to the speed estimates
    smoothed_speed = smooth_speed(speed_estimates)

    # Display smoothed speed on frame
    cv2.putText(im0_resized, f"Speed: {smoothed_speed:.2f} km/h", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Write processed frame to output video
    video_writer.write(im0_resized)

# Release resources
cap.release()
video_writer.release()
cv2.destroyAllWindows()


Ultralytics Solutions: ✅ {'region': [(20, 400), (1800, 400), (1800, 360), (20, 360)], 'show_in': True, 'show_out': True, 'colormap': None, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'json_file': None, 'records': 5, 'show': True, 'model': 'yolo11n.pt', 'classes': [2]}
WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()


0: 384x640 9 cars, 126.6ms
Speed: 5.1ms preprocess, 126.6ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 116.5ms
Speed: 3.7ms preprocess, 116.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 130.8ms
Speed: 3.7ms preprocess, 130.8ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 121.1ms
Speed: 3.6ms preprocess, 121.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 132.6ms
Speed: 3.5ms preprocess, 132.6ms inference, 1.7ms postprocess per image at shape (1